# Prepare Requirements Dataset from PURE, PROMISE, DevBench, and rSDE-Bench Sources

In this notebook, the requirements datasets that come in various formats: PDF, DOC, XML, MD are preprocessed into a common JSON format which is consumable both by the RAG index and the evaluation framework.

## Import Libraries

In [32]:
import pandas as pd
import numpy as np
import json
import re

import xmltodict
import pdfplumber as pp
from docx import Document

## Prepare a Requirements Dataset
A pandas dataset with the necessary fields to collect all the requirements in the following sections.

In [38]:
# Create an empty pandas dataframe and add the required columns to it.
requirements = pd.DataFrame()

requirements['project_id'] = pd.Series(dtype='int')
requirements['requirement'] = pd.Series(dtype='str')
requirements['tag'] = pd.Series(dtype='string')

# Create the requirements tags common to all requirements
req_tags = ['Functional', 'NonFunctional', 'Quality', 'Availability', 'FaultTolerance', 'Legal', 'LookAndFeel', 'Maintainability', 'Operability', 'Performance', 'Portability', 'Scalability', 'Security', 'Usability', 'Other']

## Preprocess the XML Files from PURE
Some of the requirements documents from PURE are available as XML files. This section uses the xmltodict and re libraries to extract the requirements and append to the requirements dataset.

In [6]:
# XML preprocessing

## Preprocess PDF Files from PURE
This block of code extracts the requirements from the PDF files in the PURE requirements dataset. The libraries used are pdfplumber, and re.

In [7]:
# PDF preprocessing

## Preprocess Docx Files from PURE
Some of the requirements in PURE are in .doc format. These files are first opened in Word and saved as .docx files since the python-docx library is better at handling .docx files. The libraries used are python-docx, and re.

In [8]:
# Docx preprocessing

## Preprocess CSV File from PROMISE
The PROMISE dataset comes in a CSV file with functional and non-functional requirements and 15 labels or tags for each requirement. These set of requirements are also appended to the JSON file after preprocessing them using the pandas library.

In [40]:
# CSV preprocessing
promise = pd.read_csv('../../datasets/promise/PROMISE-relabeled-NICE.csv')
print(promise.head())

# Add the RequirementText field to the requirement field of the requirements dataset and 
# the add the other fields that are set to 1, while also setting the Isfunctional = 0 values to NonFunctional, to the tags array

# A map to convert the promise binary field names into tags with friendly names
REQ_TYPE_MAP = {
    'IsFunctional': 'Functional',
    'IsQuality':     'Quality',
    'Availability (A)': 'Availability',
    'Fault Tolerance (FT)': 'FaultTolerance',
    'Legal (L)': 'Legal',
    'Look & Feel (LF)': 'LookAndFeel',
    'Maintainability (MN)': 'Maintainability',
    'Operability (O)': 'Operability',
    'Performance (PE)':   'Performance',
    'Portability (PO)': 'Portability',
    'Scalability (SC)': 'Scalability',
    'Security (SE)': 'Security',
    'Usability (US)': 'Usability',
    'Other (OT)': 'Other'
}

def _build_tag_row(row: pd.Series) -> list:
    """
    Return a list of friendly names for the row.

    Parameters
        One row of the promise dataframe.

    Returns
        List of friendly names for that row.
    """
    tags = []

    # Handle functional/non-functional values
    if row['IsFunctional'] == 1:
        tags.append('Functional')
    else:
        tags.append('NonFunctional')

    # Add the other flags only when they are 1
    for col, friendly in REQ_TYPE_MAP.items():
        if col == 'IsFunctional':
            continue
        if row[col] == 1:
            tags.append(friendly)

    return tags


def append_with_transform(source, destination) -> pd.DataFrame:
    """
    Append the transformed promise dataframe to the overall requirements dataframe.

    Steps performed:
      1. Rename `RequirementText` to `requirement`.
      2. Build a `tag` column that:
           • Adds 'Functional' or 'NonFunctional' depending on `IsFunctional`.
           • Adds other friendly names when the respective flag is 1.
      3. Drop the original binary columns.
      4. Concatenate the two dataframes (ignoring the original index).

    Parameters:
        source: The source dataframe
        destination: the destination dataframe

    Returns
        The concatenated dataframe.
    """

    # Strip the RequirementText field from single quotes
    source['RequirementText'] = source['RequirementText'].str.strip("'")

    # Rename fields
    source = source.rename(columns={'ProjectID': 'project_id'})
    source = source.rename(columns={'RequirementText': 'requirement'})

    # Create the tag field
    source['tag'] = source.apply(
        lambda r: _build_tag_row(r), axis=1
    )

    # Keep only the required fields in the promise dataframe
    source = source[['project_id', 'requirement', 'tag']]

    # Append the promise data to the main requirements dataframe
    return pd.concat([destination, source], ignore_index=True)

# Perform the transformation and appending
requirements = append_with_transform(promise, requirements)
print(requirements.head())


   ProjectID                                    RequirementText  IsFunctional  \
0          1  'The system shall refresh the display every 60...             1   
1          1  'The application shall match the color of the ...             0   
2          1  'If projected the data must be readable. On a ...             0   
3          1  'The product shall be available during normal ...             0   
4          1  'If projected the data must be understandable....             0   

   IsQuality  Availability (A)  Fault Tolerance (FT)  Legal (L)  \
0          1                 0                     0          0   
1          1                 0                     0          1   
2          1                 0                     0          0   
3          1                 1                     0          0   
4          1                 0                     0          0   

   Look & Feel (LF)  Maintainability (MN)  Operability (O)  Performance (PE)  \
0                 0           

## Preprocess the DevBench Dataset
The DevBench benchmarking dataset comes as a set of PRD (Product Requirements Document), Architecture, Class and Sequence diagrams. This block extracts the requirements from the PRD document and appends to the overall requirements dataset. The architecture and design files are used as is to compare agent design outputs with the DevBench designs as goldens. This block uses the pandas and re libraries for preprocessing.

## Preprocess the rSDE-Bench Dataset
The rSDE-Bench dataset contains game and website requirements in .md format. This block extracts the requirements from the documents and appends it to the overall requirements dataset. The re, and pandas libraries are used for this preprocessing.

In [12]:
# rSDE-Bench preprocessing

## Generate the Requirements JSON File
This is the final step of the preprocessing which converts the pandas dataset into JSON and writes the output to disk.

In [42]:
# Serialize the requirements dataframe to a JSON file.
output_path = '../../datasets/requirements/requirements.json'

# `indent=2` makes the file pretty‑printed.
requirements.to_json(output_path, orient='records', lines=False, indent=2)